In [ ]:
from dotenv import load_dotenv

from app.agent.graph import DEFAULT_RECURSION_LIMIT
from app.config.settings import Settings
from app.runtime import build_agent_application
from langchain_core.messages import HumanMessage

print("Loading configuration and building the agent...")
load_dotenv()
settings = Settings.from_env(require_groq=True)
app = build_agent_application(settings)
print("Agent is ready.")


In [ ]:
# The graph state, nodes, tool routing, and graph assembly live in app/agent/.
print("Workflow assembled by app.runtime and app.agent.graph.")


In [ ]:
# Notebook entry point: share the application prompt and output contract.
from pathlib import Path

from app.agent.plan_output import normalize_groq_plan, parse_and_validate_plan, render_plan_markdown
from app.agent.prompts import build_plan_prompt
from app.rag.citations import citation_registry_from_messages

USER_PROFILE = {
    "age": 28,
    "weight_kg": 60.0,
    "height_cm": 164.0,
    "frequency": 5,
    "gender": "female",
    "goal": "hipertrofia",
}

from app.tools.calculations import calculate_fitness_metrics, format_fitness_metrics

USER_PROFILE["reference_metrics"] = format_fitness_metrics(
    calculate_fitness_metrics(
        age_years=USER_PROFILE["age"],
        weight_kg=USER_PROFILE["weight_kg"],
        height_cm=USER_PROFILE["height_cm"],
        sex=USER_PROFILE["gender"],
        training_days_per_week=USER_PROFILE["frequency"],
        goal=USER_PROFILE["goal"],
    ),
    USER_PROFILE["goal"],
)

result = app.invoke(
    {
        "messages": [HumanMessage(content=build_plan_prompt(USER_PROFILE))],
        "sessions_per_week": USER_PROFILE["frequency"],
    },
    config={"recursion_limit": DEFAULT_RECURSION_LIMIT},
)
raw_content = normalize_groq_plan(
    result["messages"][-1].content,
    sessions_per_week=USER_PROFILE["frequency"],
)
citation_registry = citation_registry_from_messages(result["messages"])
plan = parse_and_validate_plan(
    raw_content,
    sessions_per_week=USER_PROFILE["frequency"],
    citation_registry=citation_registry,
)
final_plan_content = render_plan_markdown(plan, citation_registry)

output_path = Path("outputs/plano_de_treino_final.md")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(final_plan_content, encoding="utf-8")
print(final_plan_content)
print(f"Plan saved to {output_path}")
